# Question 3 — Corners

Do winning teams have a significantly higher average number of corners than losing teams?

**Objective 1.** Run the cells from top to bottom. This notebook loads the original CSV independently. Tables and graphs appear directly beneath their code cells.

## 1. Libraries and project paths

Use the project `.venv` kernel. Inline plotting keeps the figures inside this notebook.

In [1]:
# Display graphs directly below their notebook cells.
%matplotlib inline
from pathlib import Path
import pandas as pd
import numpy as np
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

QUESTION = 'Do winning teams have a significantly higher average number of corners than losing teams?'
NUMBER = 3
# Locate the project whether this notebook starts in the root or its question folder.
ROOT = Path.cwd()
if not (ROOT / 'data').is_dir():
    ROOT = ROOT.parent
BASE = ROOT / 'Question_3_Corners'
for directory in ['processed_data', 'figures', 'tables', 'results']:
    (BASE / directory).mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 35)
print(QUESTION)

Do winning teams have a significantly higher average number of corners than losing teams?


## 2. Load and inspect the raw CSV

The file has 28 columns and repeated stage/header rows. We inspect these before selecting match observations.

In [4]:
ROOT = Path.cwd()
while not (ROOT / 'data').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

path = ROOT / 'data' / 'raw' / 'World Cup - 2026 - Stats - Fixtures.csv'

# Keep every row initially because this CSV contains several header rows.
raw = pd.read_csv(path, header=None)

# Read the third row as column labels for inspection only;
# no match data are loaded here.
headers = pd.read_csv(path, header=2, nrows=0).columns.tolist()

print('Raw rows:', len(raw), '| Columns:', raw.shape[1])
display(raw.head())

Raw rows: 121 | Columns: 28


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27
0,GROUP STAGE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,GOALS,NaN,NaN,NaN,CARDS,NaN,NaN,NaN,CORNERS,NaN,NaN,NaN,NaN,NaN,NaN,xG,NaN,SHOTS,NaN,SHOTS ON\r\nTARGET,NaN,FOULS,NaN
2,Date,Team 1,Result,NaN,Team 2,1H,NaN,2H,NaN,YEL,NaN,RED,NaN,1H C,NaN,2H C,NaN,MC,NaN,T,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,11-06-26,Mexico,2,0,South Africa,1,0.0,1,0.0,1,2.0,1,2.0,2,1.0,1,0.0,3,1.0,4,1.46,0.07,16,3.0,4,2.0,12,11.0
4,12-06-26,South Korea,2,1,Czech Republic,0,0.0,2,1.0,1,0.0,0,0.0,3,3.0,1,2.0,4,5.0,9,2.3,0.83,15,7.0,6,4.0,9,16.0


In [5]:
# Combine column names, types and missing counts into one readable inspection table.
column_inspection = pd.DataFrame({
    'Position': range(raw.shape[1]),
    'Original pandas name (header=2)': headers,
    'Raw data type': raw.dtypes.astype(str).values,
    'Missing values': raw.isnull().sum().values
})
display(column_inspection)
print('Raw duplicate records (including repeated headers):', raw.duplicated().sum())

,Position,Original pandas name (header=2),Raw data type,Missing values
0,0,Date,object,6
1,1,Team 1,object,12
2,2,Result,object,12
3,3,Unnamed: 3,object,18
4,4,Team 2,object,12
5,5,1H,object,6
6,6,Unnamed: 6,float64,18
7,7,2H,object,12
8,8,Unnamed: 8,float64,18
9,9,YEL,object,6


Raw duplicate records (including repeated headers): 9


## 3. Extract match rows and identify stages

Stage labels are section headings. Carry them down to their matches and remove the repeated headers. No merge is required.

In [12]:
stages = ['GROUP STAGE', 'ROUND OF 32', 'ROUND OF 16',
          'QUARTER FINALS', 'SEMI FINALS', 'FINAL']
if raw.shape[1] != 29 or raw.iloc[2, 2] != 'Result' or raw.iloc[2, 17] != 'MC':
    raise ValueError('CSV layout changed: inspect its headers before analysing.')

# Keep recognised stage headings, then carry each heading down to its match rows.
raw['Stage'] = raw[0].where(raw[0].isin(stages)).ffill()

# A date-shaped first field identifies a match rather than a repeated header.
match_rows = raw[0].astype('str').str.fullmatch(r'\d{1,2}-\d{1,2}-\d{2}', na=False)
matches = raw.loc[match_rows].copy()

print('Match observations:', len(matches))
display(matches.head())
display(matches['Stage'].value_counts().rename('Matches').to_frame())

Match observations: 103


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,Stage
3,11-06-26,Mexico,2,0,South Africa,1,0.0,1,0.0,1,2.0,1,2.0,2,1.0,1,0.0,3,1.0,4,1.46,0.07,16,3.0,4,2.0,12,11.0,GROUP STAGE
4,12-06-26,South Korea,2,1,Czech Republic,0,0.0,2,1.0,1,0.0,0,0.0,3,3.0,1,2.0,4,5.0,9,2.3,0.83,15,7.0,6,4.0,9,16.0,GROUP STAGE
5,12-06-26,Canada,1,1,Bosnia & Herzegovina,0,1.0,1,0.0,2,3.0,0,0.0,9,1.0,0,3.0,9,4.0,13,1.25,0.98,13,8.0,4,3.0,10,20.0,GROUP STAGE
6,13-06-26,USA,4,1,Paraguay,3,0.0,1,1.0,1,5.0,0,0.0,2,0.0,1,1.0,3,1.0,4,1.42,0.54,16,9.0,6,1.0,13,17.0,GROUP STAGE
7,13-06-26,Qatar,1,1,Switzerland,0,1.0,1,0.0,2,1.0,0,0.0,3,4.0,0,6.0,3,10.0,13,0.6,3.20,6,26.0,3,7.0,12,11.0,GROUP STAGE


,Matches
Stage,
GROUP STAGE,72
ROUND OF 32,16
ROUND OF 16,8
QUARTER FINALS,4
SEMI FINALS,2
FINAL,1
